# 🧬 Pathoplexus - Genomic Pathogen Data

This notebook demonstrates how to access genomic data from **Pathoplexus**,
an open-source database for sharing human viral pathogen genomic data.

## Data Source
- **Portal**: https://pathoplexus.org/
- **API**: LAPIS (Lightweight API for Sequences)
- **Coverage**: Global (focus on human viral pathogens)
- **Integration**: INSDC databases (NCBI, ENA, DDBJ)
- **License**: Open Data (CC BY 4.0 for open data)

## Supported Organisms
- 🦟 **Dengue Virus** - Critical for Brazil
- 🦠 **Mpox Virus** - Emerging pathogen
- 🟡 **Yellow Fever Virus** - Seasonal outbreaks
- 🦠 **West Nile Virus**
- 🔴 **Measles Virus**
- And more: Ebola, Marburg, RSV, HMPV, CCHF

## API Capabilities
- 📊 Metadata queries
- 🧬 Sequence retrieval (aligned/unaligned)
- 🔄 Mutation analysis
- 📈 Aggregation counts
- 🗺️ Geographic filtering

In [ ]:
# Setup and imports
import sys
sys.path.insert(0, '../../scripts/accessors')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from pathoplexus import (
    PathoplexusAccessor,
    get_dengue_brazil,
    get_mpox_brazil,
    get_measles_brazil,
)

# Plotting setup
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Pathoplexus accessor loaded successfully!")
print(f"Current date: {datetime.now().strftime('%Y-%m-%d')}")

## 1. Explore Supported Organisms

In [ ]:
# List all supported organisms
ppx = PathoplexusAccessor('dengue')  # Initialize with any organism
organisms = ppx.list_organisms()

print("Supported Organisms in Pathoplexus:")
print("="*60)
print(organisms.to_string(index=False))
print(f"\nTotal organisms: {len(organisms)}")

## 2. Dengue Virus - Brazil Analysis

Dengue is a major public health concern in Brazil. Let's analyze the genomic data available.

In [ ]:
# Initialize dengue accessor
dengue = PathoplexusAccessor('dengue')

# Get summary for Brazil (2024)
print("Fetching dengue summary for Brazil...")
summary = dengue.get_brazil_summary(year=2024)

print("\n" + "="*60)
print("Dengue in Brazil - 2024 Summary")
print("="*60)
print(f"Total sequences: {summary['total_sequences']}")

if summary['by_serotype']:
    print("\nBy Serotype:")
    for st, count in summary['by_serotype'].items():
        pct = (count / summary['total_sequences'] * 100) if summary['total_sequences'] > 0 else 0
        print(f"  {st}: {count} ({pct:.1f}%)")

if summary['by_state']:
    print("\nTop 10 States:")
    sorted_states = sorted(summary['by_state'].items(), key=lambda x: x[1], reverse=True)[:10]
    for state, count in sorted_states:
        print(f"  {state}: {count}")

In [ ]:
# Visualize serotype distribution
if summary['by_serotype']:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    serotypes = list(summary['by_serotype'].keys())
    counts = list(summary['by_serotype'].values())
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
    bars = ax.bar(serotypes, counts, color=colors[:len(serotypes)], alpha=0.8)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_title('Dengue Serotypes in Brazil (2024)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Serotype')
    ax.set_ylabel('Number of Sequences')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Get detailed metadata for Brazil
print("Fetching detailed metadata...")
metadata = dengue.get_metadata(
    country='Brazil',
    date_from='2024-01-01',
    limit=1000,
)

print(f"\nRetrieved {len(metadata)} records")
print(f"\nColumns: {list(metadata.columns)}")
print("\nFirst few records:")
display_cols = ['accession', 'geoLocAdmin1', 'sampleCollectionDate', 'serotype', 'dataUseTerms']
print(metadata[display_cols].head(10).to_string())

In [ ]:
# Temporal analysis
if 'sampleCollectionDate' in metadata.columns and not metadata.empty:
    metadata['sampleCollectionDate'] = pd.to_datetime(metadata['sampleCollectionDate'], errors='coerce')
    metadata['month'] = metadata['sampleCollectionDate'].dt.to_period('M')
    
    # Monthly counts
    monthly = metadata.groupby('month').size()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    monthly.plot(kind='line', marker='o', ax=ax, linewidth=2, markersize=6)
    ax.set_title('Dengue Sequences Submissions by Month (2024)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Number of Sequences')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Geographic Distribution

In [ ]:
# State-level analysis
if 'geoLocAdmin1' in metadata.columns and not metadata.empty:
    state_counts = metadata['geoLocAdmin1'].value_counts().head(15)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    state_counts.plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
    
    # Add value labels
    for i, v in enumerate(state_counts.values):
        ax.text(v + 0.5, i, str(v), va='center', fontsize=10)
    
    ax.set_title('Dengue Sequences by State (Top 15) - Brazil 2024', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Sequences')
    ax.set_ylabel('State')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

## 4. Mpox Analysis

Mpox has been an emerging concern since the 2022 outbreak.

In [ ]:
# Initialize mpox accessor
mpox = PathoplexusAccessor('mpox')

# Get count for Brazil
mpox_count = mpox.count_sequences(country='Brazil', date_from='2022-01-01')
print(f"Mpox sequences from Brazil (since 2022): {mpox_count}")

# Get metadata
if mpox_count > 0:
    mpox_metadata = mpox.get_metadata(
        country='Brazil',
        date_from='2022-01-01',
        limit=1000
    )
    
    print(f"\nRetrieved {len(mpox_metadata)} records")
    
    if 'lineage' in mpox_metadata.columns:
        print("\nBy Lineage:")
        print(mpox_metadata['lineage'].value_counts().head(10))

## 5. Mutation Analysis

Analyze mutations in dengue sequences.

In [ ]:
# Get mutations for dengue in Brazil
print("Fetching mutation data...")
try:
    mutations = dengue.get_mutations(
        mutation_type='nucleotide',
        min_proportion=0.01,  # At least 1% frequency
        country='Brazil',
        limit=50
    )
    
    print(f"\nFound {len(mutations)} mutations")
    print("\nTop 10 mutations:")
    print(mutations.head(10).to_string())
except Exception as e:
    print(f"Error fetching mutations: {e}")

## 6. Data Export

In [ ]:
# Export metadata to CSV
if 'metadata' in locals() and not metadata.empty:
    output_file = "pathoplexus_dengue_brazil_2024.csv"
    metadata.to_csv(output_file, index=False)
    print(f"✅ Metadata exported to: {output_file}")
    
    from pathlib import Path
    file_size = Path(output_file).stat().st_size / 1024
    print(f"📁 File size: {file_size:.2f} KB")
    print(f"📊 Records: {len(metadata)}")

## Summary

This notebook demonstrated:

### ✅ Key Features
1. **12 supported organisms** including dengue, mpox, measles
2. **Metadata queries** with geographic and temporal filters
3. **Sequence counts** and aggregation
4. **Mutation analysis** for genomic surveillance
5. **Brazil-focused analysis** with state breakdown

### 📊 Data Highlights
- Dengue: Serotype distribution, geographic spread, temporal trends
- Mpox: Lineage tracking for outbreak analysis
- Geographic coverage: All 27 Brazilian states potential

### 🔧 Usage Patterns
- Filter by country, state, date range
- Analyze serotypes/lineages
- Track mutations over time
- Export for downstream analysis

### 📚 Resources
- Pathoplexus: https://pathoplexus.org/
- API Docs: https://pathoplexus.org/api-documentation
- GitHub: https://github.com/pathoplexus/pathoplexus
- Accessor: `scripts/accessors/pathoplexus.py`